# 실험: local 피처만 쓸 때 GNN 임베딩이 이웃 정보를 대신할 수 있는가

**배경.** Elliptic 피처 165개 = **local 93개**(0~92열, 거래 자체 정보) + **aggregated 72개**(93~164열, 직접 연결된 이웃 거래의 집계값).
aggregated 피처는 사람이 미리 만든 "1-hop 이웃 정보"입니다. 165개를 다 쓴 최종 실행(`colab_hybrid_final.ipynb`)에서는 OOF 하이브리드가 XGBoost와 동급이었는데(GCN +0.002, 3개 결합 −0.003), 원본에 이웃 정보가 이미 들어 있어서 GNN 임베딩이 더할 게 적었기 때문으로 보입니다.

**질문.** aggregated 피처를 빼고 local 93개만 쓰면, GNN 임베딩이 빠진 이웃 정보를 채워서 XGBoost(local만)보다 좋아지는가?

| 조건 | 입력 | 이웃 정보 |
|---|---|---|
| `xgb_local` | local 93 | 없음 |
| `hybrid_{gcn,sage,gat}_local` | local 93 + 그 GNN의 OOF 임베딩 64 | GNN이 학습 |
| **`hybrid_all_local`** | local 93 + 3개 OOF 임베딩 192 (285차원) | **GNN이 학습 (본 비교)** |
| `..._naive` | local 93 + 누수 있는 임베딩 | 비교용 |
| `xgb_full` | local 93 + agg 72 (=기존 기준선) | 사람이 만든 집계 |
| `hybrid_all_full` | 165 + 3개 OOF 임베딩 (최종 실행과 같은 실험, 같은 환경에서 재실행) | 둘 다 |

**미리 정한 판단 기준 (결과를 보기 전에 고정).**
- 주 비교: `hybrid_all_local` vs `xgb_local`, 테스트(35~49) PR-AUC, 같은 시드끼리 짝지은 차이.
- "개선"이라고 쓰는 조건: **5시드 모두 높고, 대응 t검정 p < 0.05**, 그리고 해석 중심 구간인 **35~42에서도 같은 방향**.
- 회복 비율 = (`hybrid_local` − `xgb_local`) / (`xgb_full` − `xgb_local`): 사람이 만든 집계 피처가 주는 이득 중 GNN 임베딩이 얼마나 되찾는지.
- 결과가 좋든 나쁘든 이 노트북의 모든 실험을 그대로 보고합니다. 설정 선택은 검증으로만, 테스트는 보고에만 씁니다.

**고정한 것.** GNN 하이퍼파라미터는 165개 피처로 정한 각 모델의 최종 설정을 그대로 씁니다(local 조건에서 다시 튜닝하지 않음). XGBoost 설정도 베이스라인 최적 설정 고정. 둘 다 보고서에 한계로 적습니다.

**런타임:** GPU(T4/L4). 예상 40~60분.

In [ ]:
# 1) 환경 확인
!python --version
!nvidia-smi -L || echo 'GPU 없음 -> 런타임 유형을 GPU로 바꾸세요'
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2) 저장소(main) 받기 + Git LFS
import getpass, os
if not os.path.exists('/content/2026-2-HAI-ProjectTeam8'):
    tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
    host = f'{tok}@' if tok else ''
    !git clone -q --depth 1 -b main https://{host}github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git /content/2026-2-HAI-ProjectTeam8
    del tok, host
%cd /content/2026-2-HAI-ProjectTeam8
!git lfs version || (apt-get -qq install git-lfs > /dev/null && git lfs version)
!git lfs install --local > /dev/null && git lfs pull
!git log --oneline -1
!ls models hybrid

In [ ]:
# 3) 데이터 복사 (드라이브 -> data/elliptic)
import os, shutil
from google.colab import drive
drive.mount('/content/drive')
CANDIDATES = ["/content/drive/MyDrive/elliptic", "/content/drive/MyDrive/Colab Notebooks/elliptic"]
NEED = ["elliptic_txs_features.csv", "elliptic_txs_classes.csv", "elliptic_txs_edgelist.csv"]
src = next((p for p in CANDIDATES if all(os.path.exists(f"{p}/{f}") for f in NEED)), None)
if src is None:
    for p in CANDIDATES:
        print(p, "->", os.listdir(p) if os.path.isdir(p) else "없음")
    raise FileNotFoundError("elliptic CSV 3개를 못 찾음")
os.makedirs("data/elliptic", exist_ok=True)
for f in NEED:
    shutil.copy(f"{src}/{f}", f"data/elliptic/{f}")
print("복사 완료:", src)

In [ ]:
# 4) 설치
!pip install -q torch-geometric "xgboost>=3.4.1,<3.5" scipy
import xgboost, torch_geometric
print('xgboost', xgboost.__version__, '| torch_geometric', torch_geometric.__version__)

## 5. local 전용 래퍼
`hybrid/local_only.py`: `load_data()`가 돌려주는 피처를 앞 93개(local)로 자른 뒤 기존 스크립트(`models.*`, `hybrid.make_oof_emb`, `hybrid.run_hybrid`)를 **수정 없이** 그대로 실행합니다. 저장소에 아직 없으면 아래 셀이 파일을 만듭니다 (이미 있으면 그 파일과 같은 내용).

In [ ]:
%%writefile hybrid/local_only.py
"""local 피처(93개)만 쓰는 조건으로 기존 스크립트를 그대로 실행하는 래퍼.

Elliptic 피처 165개 = local 93개(0~92열, 거래 자체 정보) + aggregated 72개(93~164열, 1-hop 이웃 집계).
aggregated 피처는 사람이 미리 만든 '이웃 정보'라서, 165개를 다 쓰면 GNN 임베딩이 더할 정보가 적다.
이 래퍼는 load_data()가 돌려주는 피처를 앞 93개로 자른 뒤 지정한 모듈을 실행한다.
GNN 학습(models.*), OOF 임베딩(hybrid.make_oof_emb), 하이브리드(hybrid.run_hybrid) 코드는 수정하지 않는다.

  python -m hybrid.local_only models.gcn --data-root data/elliptic --results-dir results_local --lr 0.01 ...
  python -m hybrid.local_only hybrid.make_oof_emb --model gcn --ref-emb-dir results_local/emb ...
  python -m hybrid.local_only hybrid.run_hybrid --mode raw --name xgb_local --results-dir results_local

결과 이름·폴더는 기존 결과와 섞이지 않게 실행할 때 따로 지정한다 (--results-dir, --name, --out-dir).
"""
from __future__ import annotations

import dataclasses
import runpy
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import common.data as cdata  # noqa: E402

N_LOCAL = 93
_orig_load_data = cdata.load_data


def load_data_local(*args, **kwargs):
    """common.data.load_data와 같고, x와 x_std를 local 93열로 자른다 (표준화 통계는 열별이라 그대로 유효)."""
    d = _orig_load_data(*args, **kwargs)
    if d.x.shape[1] != cdata.EXPECTED["n_features"]:
        raise ValueError(f"피처 수가 {d.x.shape[1]}개입니다. 165개(원본)여야 local 93개를 정확히 고를 수 있습니다.")
    x_std = None if d.x_std is None else d.x_std[:, :N_LOCAL].copy()
    return dataclasses.replace(d, x=d.x[:, :N_LOCAL].copy(), x_std=x_std)


def main(argv=None):
    argv = list(sys.argv[1:] if argv is None else argv)
    if not argv or argv[0].startswith("-"):
        raise SystemExit("사용법: python -m hybrid.local_only <모듈> [그 모듈의 인자...]  (예: models.gcn, hybrid.run_hybrid)")
    module, rest = argv[0], argv[1:]
    # 대상 모듈이 `from common.data import load_data`로 가져가기 전에 바꿔 둔다
    cdata.load_data = load_data_local
    for name, mod in list(sys.modules.items()):
        if name != "common.data" and getattr(mod, "load_data", None) is _orig_load_data:
            mod.load_data = load_data_local
    print(f"[local_only] 피처를 local {N_LOCAL}개로 제한해서 실행: {module} {' '.join(rest)}", flush=True)
    sys.argv = [module, *rest]
    runpy.run_module(module, run_name="__main__", alter_sys=True)


if __name__ == "__main__":
    main()

In [ ]:
# 6) 설정 + 래퍼 점검: local 93열이 맞는지, GNN 입력 차원이 93인지
import numpy as np, pandas as pd, subprocess, time, importlib
from scipy import stats
pd.set_option('display.width', 230)
RL = 'results_local'                       # 이 노트북의 모든 결과
SEEDS = [0, 1, 2, 3, 4]
GNNS = ['gcn', 'sage', 'gat']
GNN_CFG = {                                # 165개 피처로 정한 각 모델의 최종 설정 (그대로 고정)
    'gcn':  dict(lr=0.01,  dropout=0.5, wd=5e-4),
    'sage': dict(lr=0.005, dropout=0.3, wd=1e-3),
    'gat':  dict(lr=0.01,  dropout=0.3, wd=5e-4),   # heads 8 (models/gat.py 기본값)
}
import hybrid.local_only as lo
from common.data import load_data
full, loc = lo._orig_load_data('data/elliptic', standardize=True), lo.load_data_local('data/elliptic', standardize=True)
assert full.x.shape[1] == 165 and loc.x.shape[1] == 93 and loc.x_std.shape[1] == 93
assert np.array_equal(full.x[:, :93], loc.x) and np.array_equal(full.y, loc.y) and np.array_equal(full.edge_index, loc.edge_index)
print('local 93열 확인 OK | 노드', loc.num_nodes, '| 라벨 분할 동일')
del full, loc

def sh(cmd):
    print('$', ' '.join(cmd)); t0 = time.time()
    out = subprocess.run(cmd, capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout[-3000:]); print(out.stderr[-3000:]); raise RuntimeError('실패: ' + ' '.join(cmd))
    return out.stdout, time.time() - t0

## 7. GNN 단독 학습 (local 93개 입력)
각 모델을 local 피처만으로 처음부터 다시 학습합니다 (공통 절차 그대로: Train 학습 + 검증 조기 종료 → Train+Val 재학습 → 테스트 1회, patience 100, 최대 500 에폭).
이 실행의 임베딩이 OOF 정렬의 **기준 임베딩**이 되고, 이 실행의 best_epoch가 OOF 학습 에폭이 됩니다 (임베딩·지표·에폭이 모두 같은 실행).

In [ ]:
# 7) GNN 단독 (local) -> results_local/{metrics,pred,emb}/..., timestep_f1_{model}.csv
gnn_sec = {}
for m in GNNS:
    c = GNN_CFG[m]
    out, sec = sh(['python', '-m', 'hybrid.local_only', f'models.{m}', '--data-root', 'data/elliptic', '--results-dir', RL,
                   '--seeds', *map(str, SEEDS), '--lr', str(c['lr']), '--dropout', str(c['dropout']), '--weight-decay', str(c['wd']),
                   '--patience', '100', '--max-epochs', '500'])
    gnn_sec[m] = sec
    print('\n'.join(l for l in out.splitlines() if l.startswith('[seed')), f'\n== {m}: {sec / 60:.1f}분\n')

rows = []
for m in GNNS:
    def t(path):
        d = pd.read_csv(path, dtype={'threshold_type': str})
        return d[(d.split == 'test') & (d.threshold_type == 'fixed_0.5')].sort_values('seed')
    tl, tf = t(f'{RL}/metrics/{m}.csv'), t(f'results/metrics/{m}.csv')
    rows.append(dict(모델=m, **{'local 93 PR-AUC': f'{tl.pr_auc.mean():.4f} ± {tl.pr_auc.std():.4f}',
                              '165개 PR-AUC (저장소)': f'{tf.pr_auc.mean():.4f} ± {tf.pr_auc.std():.4f}',
                              'local ROC-AUC': round(tl.roc_auc.mean(), 4), 'n_params': int(tl.n_params.iloc[0]),
                              'best_epoch': tl.notes.str.extract(r'best_epoch=(\d+)')[0].tolist()}))
print(pd.DataFrame(rows).to_string(index=False))

## 8. OOF 임베딩 (local GNN)
최종 실행과 같은 팀 표준 방식: 시간 블록 5개, 블록마다 나머지 블록 정답으로 학습, 라벨 없는 노드로 기준 임베딩(7번의 local GNN)에 Procrustes 정렬, 테스트는 기준 임베딩 그대로.

In [ ]:
# 8) OOF 임베딩 -> results_local/emb_oof/emb_oof_{model}_seed{n}.npy + oof_report_{model}.csv
oof_sec = {}
for m in GNNS:
    c = GNN_CFG[m]
    out, sec = sh(['python', '-m', 'hybrid.local_only', 'hybrid.make_oof_emb', '--model', m, '--data-root', 'data/elliptic',
                   '--lr', str(c['lr']), '--dropout', str(c['dropout']), '--weight-decay', str(c['wd']), '--seeds', *map(str, SEEDS),
                   '--metrics-csv', f'{RL}/metrics/{m}.csv', '--ref-emb-dir', f'{RL}/emb', '--out-dir', f'{RL}/emb_oof'])
    oof_sec[m] = sec
    print('\n'.join(l for l in out.splitlines() if l.startswith('[seed')), f'\n== {m}: {sec / 60:.1f}분\n')

diag = []
for m in GNNS:
    r = pd.read_csv(f'{RL}/emb_oof/oof_report_{m}.csv')
    diag.append(dict(model=m, cos_raw=r.cos_raw.mean(), cos_aligned_labeled=r.cos_aligned_labeled.mean(),
                     oof_pr_auc_pooled=r.groupby('seed').oof_pr_auc_pooled.first().mean(),
                     fold1_oof_pr_auc=r[r.fold == 1].oof_pr_auc.mean()))
print(pd.DataFrame(diag).round(3).to_string(index=False))

## 9. XGBoost 실험
`*_local`은 래퍼로 실행(원본 = local 93개), `xgb_full`·`hybrid_all_full`은 래퍼 없이 실행(원본 165개). XGBoost 설정은 모두 베이스라인 최적값 고정.

`hybrid_all_full`은 최종 실행(`colab_hybrid_final.ipynb`)이 드라이브 `MyDrive/hybrid_final/emb_oof`에 백업한 165개 피처 GNN의 OOF 임베딩을 씁니다. 없으면 건너뜁니다.

In [ ]:
# 9) XGBoost 하이브리드
FULL_OOF = '/content/drive/MyDrive/hybrid_final/emb_oof'
have_full_oof = all(os.path.exists(f'{FULL_OOF}/emb_oof_{m}_seed{s}.npy') for m in GNNS for s in SEEDS)
print('165개 피처 OOF 임베딩(드라이브):', '있음' if have_full_oof else '없음 -> hybrid_all_full 건너뜀')

LOCAL = ['python', '-m', 'hybrid.local_only', 'hybrid.run_hybrid']
FULL = ['python', '-m', 'hybrid.run_hybrid']
EXPS = [('xgb_local', LOCAL, ['--mode', 'raw'])]
for m in GNNS:
    EXPS += [(f'hybrid_{m}_local_naive', LOCAL, ['--emb-models', m, '--emb-dir', f'{RL}/emb']),
             (f'hybrid_{m}_local', LOCAL, ['--emb-models', m, '--emb-kind', 'oof', '--emb-dir', f'{RL}/emb_oof'])]
EXPS += [('hybrid_all_local_naive', LOCAL, ['--emb-models', *GNNS, '--emb-dir', f'{RL}/emb']),
         ('hybrid_all_local', LOCAL, ['--emb-models', *GNNS, '--emb-kind', 'oof', '--emb-dir', f'{RL}/emb_oof']),
         ('xgb_full', FULL, ['--mode', 'raw'])]
if have_full_oof:
    EXPS += [('hybrid_all_full', FULL, ['--emb-models', *GNNS, '--emb-kind', 'oof', '--emb-dir', FULL_OOF])]

for name, base_cmd, a in EXPS:
    out, sec = sh([*base_cmd, *a, '--name', name, '--data-root', 'data/elliptic', '--seeds', *map(str, SEEDS), '--results-dir', RL])
    assert f'결과 이름: {name}' in out
    print('\n'.join(l for l in out.splitlines() if l.startswith(('[seed', 'PR-AUC'))), f'\n-- {name}: {sec:.0f}초\n')
NAMES = [n for n, _, _ in EXPS]

## 10~12. 결과
주 비교는 **`hybrid_*_local` vs `xgb_local`** (같은 시드끼리). 판단 기준은 맨 위에 미리 적은 대로입니다.

In [ ]:
# 10) 테스트 PR-AUC + 짝 비교 + 회복 비율
def pick(name, split='test', tt='fixed_0.5'):
    m = pd.read_csv(f'{RL}/metrics/{name}.csv', dtype={'threshold_type': str})
    return m[(m.split == split) & (m.threshold_type == tt)].sort_values('seed').set_index('seed')

by_seed = pd.DataFrame({n: pick(n).pr_auc for n in NAMES})
print('== 테스트 PR-AUC (시드별) =='); print(by_seed.round(4).to_string()); print()

xl, xf = pick('xgb_local').pr_auc, pick('xgb_full').pr_auc
agg_gain = (xf - xl).mean()
print(f'사람이 만든 집계 피처의 이득 (xgb_full − xgb_local): {agg_gain:+.4f}\n')
rows = []
for n in NAMES:
    t, v = pick(n), pick(n, 'val', 'val_opt')
    r = {'실험': n, 'test PR-AUC': f'{t.pr_auc.mean():.4f} ± {t.pr_auc.std():.4f}', 'val PR-AUC': round(v.pr_auc.mean(), 4),
         'test ROC-AUC': round(t.roc_auc.mean(), 4)}
    if n.endswith('_local') and n != 'xgb_local':
        d = (t.pr_auc - xl).values
        r.update({'Δ vs xgb_local': f'{d.mean():+.4f} ± {d.std(ddof=1):.4f}', '높은 시드': f'{int((d > 0).sum())}/5',
                  'p': f'{stats.ttest_rel(t.pr_auc, xl).pvalue:.3f}', '회복 비율': f'{d.mean() / agg_gain:.0%}' if agg_gain > 0 else '-'})
    rows.append(r)
summary = pd.DataFrame(rows).fillna('-')
print(summary.to_string(index=False))

d = (pick('hybrid_all_local').pr_auc - xl).values
ok = (d > 0).all() and stats.ttest_rel(pick('hybrid_all_local').pr_auc, xl).pvalue < 0.05
print(f'\n[미리 정한 기준] hybrid_all_local vs xgb_local (35~49): 5시드 모두 높음={bool((d > 0).all())}, p<0.05={stats.ttest_rel(pick("hybrid_all_local").pr_auc, xl).pvalue < 0.05} -> {"충족" if ok else "미충족"} (35~42는 12번 셀에서 확인)')

In [ ]:
# 11) F1 / 정밀도 / 재현율
rows = []
for n in NAMES:
    for tt in ('fixed_0.5', 'val_opt'):
        t = pick(n, 'test', tt)
        rows.append(dict(실험=n, 임계값=tt, F1=f'{t.f1.mean():.3f} ± {t.f1.std():.3f}',
                         정밀도=round(t.precision.mean(), 3), 재현율=round(t.recall.mean(), 3)))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# 12) 구간 분리 PR-AUC (35~42 / 43~49) + gain 비중
from sklearn.metrics import average_precision_score as AP
d = load_data('data/elliptic', standardize=False)
SEG = {'35~42': (35, 42), '43~49': (43, 49)}
masks = {k: d.test_mask & (d.time_step >= a) & (d.time_step <= b) for k, (a, b) in SEG.items()}
seg_val, seg_rows = {}, []
for n in NAMES:
    r = {'실험': n}
    for k, m in masks.items():
        v = np.array([AP(d.y[m], np.load(f'{RL}/pred/pred_{n}_seed{s}.npy')[m]) for s in SEEDS])
        seg_val[n, k] = v
        r[f'PR-AUC {k}'] = f'{v.mean():.4f} ± {v.std(ddof=1):.4f}'
    if n.endswith('_local') and n != 'xgb_local':
        dd = seg_val[n, '35~42'] - seg_val['xgb_local', '35~42']
        r['Δ 35~42 vs xgb_local'] = f'{dd.mean():+.4f} ± {dd.std(ddof=1):.4f} ({int((dd > 0).sum())}/5시드 높음)'
    imp = pd.read_csv(f'{RL}/hybrid_importance_{n}.csv').groupby('group').gain_share.mean()
    r['gain 비중'] = ' / '.join(f'{g} {v:.0%}' for g, v in imp.items())
    seg_rows.append(r)
seg = pd.DataFrame(seg_rows).fillna('-')
print(seg.to_string(index=False))

final = summary.merge(seg, on='실험')
final.to_csv(f'{RL}/local_summary.csv', index=False)
by_seed.to_csv(f'{RL}/local_test_prauc_by_seed.csv')
print('\n저장:', f'{RL}/local_summary.csv')

In [ ]:
# 13) 환경 기록 + 드라이브 백업 + 작은 결과 zip (임베딩 npy 제외)
import datetime
from google.colab import files
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
with open(f'{RL}/local_env.txt', 'w') as f:
    f.write(f'local 93 피처 실험 환경\n- 생성일: {datetime.date.today()} / 저장소 main {commit} (+ hybrid/local_only.py)\n- GPU: {gpu}\n'
            f'- torch {torch.__version__} / torch_geometric {torch_geometric.__version__} / xgboost {xgboost.__version__}\n'
            f'- GNN 설정: {GNN_CFG}, patience 100, 최대 500 에폭\n- GNN 단독 소요(분): { {k: round(v / 60, 1) for k, v in gnn_sec.items()} }\n'
            f'- OOF 소요(분): { {k: round(v / 60, 1) for k, v in oof_sec.items()} }\n- hybrid_all_full 포함: {have_full_oof}\n')
print(open(f'{RL}/local_env.txt').read())
shutil.copy('hybrid/local_only.py', f'{RL}/local_only.py')
dst = '/content/drive/MyDrive/hybrid_local'
shutil.copytree(RL, dst, dirs_exist_ok=True)
print('드라이브 백업:', dst)
shutil.copytree(RL, 'out_local_small', ignore=shutil.ignore_patterns('emb_*.npy'), dirs_exist_ok=True)
shutil.make_archive('hybrid_local_small', 'zip', 'out_local_small')
files.download('hybrid_local_small.zip')